# Task MP-FC-007: First Price-Only Machine Learning Forecasting Benchmark (Ridge & HistGradientBoosting)
**Repository:** `ankit-choubey/Mandi_Prediction`  
**Execution Environment:** Kaggle CPU Kernel (Internet Disabled)  
**Evaluation Folds:** F1 (Train <= 2024-12-31, Test 2025-01-08..2025-06-30), F2 (Train <= 2025-06-30, Test 2025-07-08..2025-10-31), F3 (Train <= 2025-12-18, Test 2025-12-26..2026-03-31), F4 (Train <= 2026-03-31, Test 2026-04-08..2026-09-30)  
**Candidate Models:** Ridge Regression (L2 with median imputer + scaler) and HistGradientBoostingRegressor (native NaN)  
**Baselines Rescored:** Naive $H=7$, Moving Average (7 obs in 30d, min 3), Seasonal Naive (357d +/- 3d), Damped Drift (anchor 14d +/- 3d)  
**Target & Response:** Additive price change $\\Delta = y_{\\text{target}} - y_t$, reconstructed $\\hat{y} = y_t + \\hat{\\Delta}$. Non-positive forecasts abstain with `NON_POSITIVE_RECONSTRUCTED_FORECAST`.  
**Paired Naive Skill Score:** $\\text{Skill}_m = 1 - \\frac{\\sum |y - \\hat{y}_m|}{\\sum |y - \\hat{y}_{\\text{naive}}|}$ on common non-abstained finite subset with series-cluster bootstrap 95% CIs.


In [ ]:
# Cell 1: Environment, System Versions, Payload Decoding & Input Discovery
import os
import sys
import glob
import json
import base64
import hashlib
import time
import datetime
import math
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib

print("=== ENVIRONMENT & LIBRARY VERSIONS ===")
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sklearn: {sklearn.__version__}")

np.random.seed(42)
print("Random seed initialized to 42.")

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
src_dir = working_root / "src"
ref_dir = working_root / "data" / "reference"
cfg_dir = working_root / "experiments" / "configs"
reports_fc007_dir = working_root / "experiments" / "reports" / "FC007"
outputs_fc007_dir = working_root / "experiments" / "outputs" / "FC007"
models_dir = working_root / "models"

for d in [src_dir, ref_dir, cfg_dir, reports_fc007_dir, outputs_fc007_dir, models_dir]:
    d.mkdir(parents=True, exist_ok=True)

if not (src_dir / "__init__.py").exists():
    (src_dir / "__init__.py").write_text("")

if str(working_root.resolve()) not in sys.path:
    sys.path.insert(0, str(working_root.resolve()))
if "/kaggle/working" not in sys.path and os.path.exists("/kaggle"):
    sys.path.insert(0, "/kaggle/working")

payload_specs = {
  "market_identity.py": {
    "content_b64": "...",
    "size_bytes": 32138,
    "sha256": "85ce73c9edaf89bb5035d8edb278bf0481db1c8f65ccedd66943cbacb6609864"
  },
  "data_pipeline.py": {
    "content_b64": "...",
    "size_bytes": 39484,
    "sha256": "c542f8c571fe6eb13fc0c9eb9eb0583c739db53d6ecdcc68b8dda1cd4e4fcb5f"
  },
  "baselines.py": {
    "content_b64": "...",
    "size_bytes": 44845,
    "sha256": "9cf56e5345235bc593f3bdf62df90e39a56be794edfb5e36fed1b94588c20718"
  },
  "forecast_features.py": {
    "content_b64": "...",
    "size_bytes": 27871,
    "sha256": "d9b55d675db7ae8266ec0ed506109c46a24d04d07b34c6c764a4fab38f062cc9"
  },
  "forecast_benchmark.py": {
    "content_b64": "...",
    "size_bytes": 29949,
    "sha256": "385163bb0bb0a0fe383f7d8abae57d65ce57e8e0f825ab2dcdd8d5d092d7708d"
  },
  "market_aliases.csv": {
    "content_b64": "ZGlzdHJpY3QsbWFya2V0X3JhdyxtYXJrZXRfY2Fub25pY2FsLHJ1bGUsZXZpZGVuY2UsZmlyc3RfZGF0ZSxsYXN0X2RhdGUsb3ZlcmxhcF9kYXlzLHByaWNlX2FncmVlbWVudCxzdGF0dXMKR2hhemlhYmFkLEdoYXppYWJhZCxHaGF6aWFiYWQsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxHaGF6aWFiYWQgQVBNQyxHaGF6aWFiYWQsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApHaGF6aWFiYWQsSGFwdXIsSGFwdXIsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA1LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxIYXB1ciBBUE1DLEhhcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz01IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNSwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIsTXVyYWRuYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIgQVBNQyxNdXJhZG5hZ2FyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKR2hhemlhYmFkLE5vaWRhLE5vaWRhLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDIsMjAyNS0xMS0wMywwLCxBQ0NFUFRFRApHaGF6aWFiYWQsTm9pZGEgQVBNQyxOb2lkYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MyA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsLEFDQ0VQVEVECkpoYW5zaSxCYXJ1d2FzYWdhcixCYXJ1d2FzYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAyLDIwMjUtMTEtMDUsMCwsQUNDRVBURUQKSmhhbnNpLEJhcnV3YXNhZ2FyIEFQTUMsQmFydXdhc2FnYXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTMgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDMsMjAyNi0wOS0zMCwzLCxBQ0NFUFRFRApKaGFuc2ksQ2hpcmdhb24sQ2hpcmdhb24sSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxDaGlyZ2FvbiBBUE1DLENoaXJnYW9uLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0zIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAyLDIwMjYtMDktMzAsMywsQUNDRVBURUQKSmhhbnNpLEd1cnVzYXJhaSxHdXJ1c2FyYWksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxHdXJ1c2FyYWkgQVBNQyxHdXJ1c2FyYWksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpLEpoYW5zaSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSBBUE1DLEpoYW5zaSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkpoYW5zaSxKaGFuc2kgKEdyYWluKSxKaGFuc2kgKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDI0LTA1LTI1LDIwMjUtMTEtMDMsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSAoR3JhaW4pIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTIgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDIsMjAyNi0wMS0xNCwyLCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpIEdyYWluIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjYtMDEtMDEsMjAyNi0wMS0xNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1cixNYXVyYW5pcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1ciBBUE1DLE1hdXJhbmlwdXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApKaGFuc2ksTW90aCxNb3RoLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTW90aCBBUE1DLE1vdGgsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApLYW5wdXIsQmFyaXBhYWwgQVBNQyxCYXJpcGFhbCBBUE1DLElkZW50aXR5IChzaW5nbGUgdG9rZW4pLFVuaXF1ZSBtYXJrZXQgZm9yIG5vcm1hbGl6ZWQgdG9rZW4gaW4gZGlzdHJpY3QsMjAyNi0wMy0wMSwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkthbnB1cixDaG91YmVwdXIsQ2hvdWJlcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApLYW5wdXIsQ2hvdWJlcHVyIEFQTUMsQ2hvdWJlcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLEthbnB1cihHcmFpbiksS2FucHVyKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxLYW5wdXIoR3JhaW4pLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLFV0dGFyaXB1cmEsVXR0YXJpcHVyYSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTEwLTI0LDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLFV0dGFyaXB1cmEgQVBNQyxVdHRhcmlwdXJhLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLFZhcmlwYWFsLFZhcmlwYWFsLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApLYW5wdXIsVmFyaXBhYWwgQVBNQyxWYXJpcGFhbCwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA0LTI2LDQsLEFDQ0VQVEVECkx1Y2tub3csQmFudGhhcmEsQmFudGhhcmEsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMi0wNS0xOCwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkx1Y2tub3csQmFudGhhcmEgQVBNQyxCYW50aGFyYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkx1Y2tub3csTHVja25vdyxMdWNrbm93LElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApMdWNrbm93LEx1Y2tub3cgQVBNQyxMdWNrbm93LCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0zIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAyLDIwMjYtMDktMzAsMywsQUNDRVBURUQKVW5uYW8sQmFuZ2FybWF1LEJhbmdhcm1hdSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sQmFuZ2FybWF1IEFQTUMsQmFuZ2FybWF1LCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKVW5uYW8sUHVyd2EsUHVyd2EsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTA3LTMxLDAsLEFDQ0VQVEVEClVubmFvLFB1cndhIEFQTUMsUHVyd2EsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjYtMDItMjEsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApVbm5hbyxVbm5hbyxVbm5hbyxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sVW5uYW8gQVBNQyxVbm5hbywiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkthbnB1cixCYXJpcGFhbCBBUE1DLFZhcmlwYWFsLFJldmlldyBQZW5kaW5nIChjcm9zcy1waG9uZXRpYyBiL3YpLE5vcm1hbGl6ZWQgbWlzbWF0Y2ggKGJhcmlwYWFsIHZzIHZhcmlwYWFsKTsgY2FuZGlkYXRlIGZvciBtYW51YWwgdmVyaWZpY2F0aW9uLDIwMjYtMDMtMDEsMjAyNi0wOS0zMCwwLCxVTk1BUFBFRC9SRVZJRVcK",
    "size_bytes": 6546,
    "sha256": "57e49682f62e6e9a089aea794626758efbd00eb9446760b8e6fa489fcc301864"
  },
  "price_forecast_v1.json": {
    "content_b64": "ewogICJzY2hlbWFfdmVyc2lvbiI6ICIxLjAuMCIsCiAgImNvbnRyYWN0X2lkIjogIlBSSUNFX0ZPUkVDQVNUX1YxX1NQRUNJRklDQVRJT04iLAogICJ0YXNrX2lkIjogIk1QLUZDLTAwNyIsCiAgImdlbmVyYXRlZF9kYXRlX2lzdCI6ICIyMDI2LTExLTEyIiwKICAic3RhdHVzIjogIkFQUFJPVkVEX1NJTkdMRV9LQUdHTEVfQ1BVX1JVTiIsCiAgImV4ZWN1dGlvbl9ub3RlcyI6ICJTaW5nbGUgYXV0aG9yaXplZCBLYWdnbGUgQ1BVIHJ1biBvbiBleGlzdGluZyBkYXRhc2V0IHRoZWFua2l0Y2hvdWJleS9tYW5kaS1wcmVkaWN0aW9uLWF1ZGl0LTAwM2IgKHYxKS4gWmVybyBsb2NhbCBtb2RlbCBmaXR0aW5nIG9yIGZ1bGwtcGFuZWwgam9pbnMuIiwKICAiaW5wdXRfZGF0YXNldCI6IHsKICAgICJuYW1lIjogIm1hbmRpXzV5ZWFyX2xlZ2FjeV9wYW5lbF92MiIsCiAgICAic2hhMjU2IjogIjg1MWU3MTg0NjhkNmI4YzlmZDA3OTQ0NTM1YTQ2MjMzOGU3MWFhMTA1YWYyNTMzOWZlOTQzOGI2OTcwODhmMzEiLAogICAgInJvd19jb3VudCI6IDQ3MjM4NSwKICAgICJyb3dfY29uc2VydmF0aW9uIjogIjQ3MjM4NSA9IDQ2MzIxNyArIDg1NSArIDI5MzYgKyA1MzUwICsgMjciLAogICAgInNvdXJjZV9maWxlIjogIjUgeWVhciBkYXRhc2V0LyouY3N2IChyZWdlbmVyYXRlZCB2aWEgZGF0YV9waXBlbGluZS5weSkiLAogICAgIm5vdGVzIjogIlByaW1hcnkgaW5wdXQgaXMgcmVnZW5lcmF0ZWQgbGVnYWN5IHBhbmVsIFYyLiBDYW5kaWRhdGUgdjIuMSByZW1haW5zIHNlcGFyYXRlOyBubyBzZWNvbmQgcGFuZWwvZXhwZXJpbWVudCBpbiB0aGlzIHJ1bi4iCiAgfSwKICAibWFya2V0X2FsaWFzZXMiOiB7CiAgICAiZmlsZSI6ICJkYXRhL3JlZmVyZW5jZS9tYXJrZXRfYWxpYXNlcy5jc3YiLAogICAgInBvbGljeSI6ICJFbWJlZCBpbW11dGFibGUgbGVnYWN5IG1hcmtldF9hbGlhc2VzLmNzdiBzZXBhcmF0ZWx5IGZyb20gY2FuZGlkYXRlIHBvbGljeSBhbmQgcHJlc2VydmUgYnl0ZXMvc2NoZW1hL29yZGVyIG5lZWRlZCBmb3IgaGFzaC4iLAogICAgImRpc3B1dGVkX2tleXNfY291bnQiOiA3NwogIH0sCiAgImZvY3VzX2Nyb3BzIjogWwogICAgIldoZWF0IiwKICAgICJQb3RhdG8iLAogICAgIk9uaW9uIiwKICAgICJUb21hdG8iLAogICAgIlBhZGR5KERoYW4pKENvbW1vbikiLAogICAgIlJpY2UiCiAgXSwKICAiZm9jdXNfY3JvcHNfbm90ZXMiOiAiRm9jdXMgY3JvcHMgZXhhY3RseSBXaGVhdCwgUG90YXRvLCBPbmlvbiwgVG9tYXRvLCBQYWRkeShEaGFuKShDb21tb24pLCBSaWNlLiBNdXN0YXJkIGFuZCBCYXNtYXRpIG5vdCBzaWxlbnRseSBhZGRlZC4gTWlzc2luZyBzbmFwc2hvdCByb3cgbWVhbnMgYWJzZW50IGZyb20gc25hcHNob3QsIG5vdCBubyB0cmFkaW5nLiBEbyBub3QgZXF1YXRlIFBhZGR5KENvbW1vbikvUmljZSB3aXRob3V0IGV4cGxpY2l0IGlkZW50aXR5IGV2aWRlbmNlLiIsCiAgInRhcmdldF9zcGVjaWZpY2F0aW9uIjogewogICAgImhvcml6b25fZGF5cyI6IDcsCiAgICAidGFyZ2V0X3dpbmRvd19vZmZzZXRzIjogWzYsIDhdLAogICAgInNlbGVjdGlvbl9wb2xpY3kiOiAiTkVBUkVTVF9PQlNFUlZFRF9XSVRIX0VBUkxJRVJfVElFX0JSRUFLIiwKICAgICJleGFjdF90N19zZW5zaXRpdml0eV9yZXF1aXJlZCI6IHRydWUsCiAgICAibWF4X3RhcmdldF9kYXRlX3J1bGUiOiAidGFyZ2V0X2RhdGUgPD0gZm9sZF90ZXN0X2VuZCIsCiAgICAidHJhaW5fbWF0dXJpdHlfcnVsZSI6ICJ0YXJnZXRfZGF0ZSA8PSB0cmFpbl9jdXRvZmYgKG9yaWdpbiB0IDw9IHRyYWluX2N1dG9mZiAtIDggZGF5cyBjb25zZXJ2YXRpdmUgY2FwKSIsCiAgICAicmVzcG9uc2VfdHJhbnNmb3JtYXRpb24iOiB7CiAgICAgICJ0eXBlIjogIkFERElUSVZFX1BSSUNFX0NIQU5HRSIsCiAgICAgICJmb3JtdWxhIjogInRhcmdldF9kZWx0YSA9IHlfdGFyZ2V0IC0geV90IiwKICAgICAgInJlY29uc3RydWN0ZWRfZm9yZWNhc3QiOiAieV9oYXQgPSB5X3QgKyB0YXJnZXRfZGVsdGEiLAogICAgICAibm9uX3Bvc2l0aXZlX2Fic3RlbnRpb25fcmVhc29uIjogIk5PTl9QT1NJVElWRV9SRUNPTlNUUlVDVEVEX0ZPUkVDQVNUIiwKICAgICAgImNsaXBwaW5nX3BvbGljeSI6ICJOT19DTElQUElORyIKICAgIH0KICB9LAogICJ0ZW1wb3JhbF9mb2xkcyI6IFsKICAgIHsKICAgICAgImZvbGRfaWQiOiAiRjEiLAogICAgICAidHJhaW5fY3V0b2ZmIjogIjIwMjQtMTItMzEiLAogICAgICAidGVzdF9zdGFydCI6ICIyMDI1LTAxLTA4IiwKICAgICAgInRlc3RfZW5kIjogIjIwMjUtMDYtMzAiLAogICAgICAiZW1iYXJnb19kYXlzIjogNywKICAgICAgIm1heF90cmFpbl9vcmlnaW4iOiAiMjAyNC0xMi0yMyIsCiAgICAgICJyZWdpbWUiOiAiSElTVE9SSUNBTCIKICAgIH0sCiAgICB7CiAgICAgICJmb2xkX2lkIjogIkYyIiwKICAgICAgInRyYWluX2N1dG9mZiI6ICIyMDI1LTA2LTMwIiwKICAgICAgInRlc3Rfc3RhcnQiOiAiMjAyNS0wNy0wOCIsCiAgICAgICJ0ZXN0X2VuZCI6ICIyMDI1LTEwLTMxIiwKICAgICAgImVtYmFyZ29fZGF5cyI6IDcsCiAgICAgICJtYXhfdHJhaW5fb3JpZ2luIjogIjIwMjUtMDYtMjIiLAogICAgICAicmVnaW1lIjogIlRSQU5TSVRJT04iCiAgICB9LAogICAgewogICAgICAiZm9sZF9pZCI6ICJGMyIsCiAgICAgICJ0cmFpbl9jdXRvZmYiOiAiMjAyNS0xMi0xOCIsCiAgICAgICJ0ZXN0X3N0YXJ0IjogIjIwMjUtMTItMjYiLAogICAgICAidGVzdF9lbmQiOiAiMjAyNi0wMy0zMSIsCiAgICAgICJlbWJhcmdvX2RheXMiOiA3LAogICAgICAibWF4X3RyYWluX29yaWdpbiI6ICIyMDI1LTEyLTEwIiwKICAgICAgInJlZ2ltZSI6ICJUUkFOU0lUSU9OIgogICAgfSwKICAgIHsKICAgICAgImZvbGRfaWQiOiAiRjQiLAogICAgICAidHJhaW5fY3V0b2ZmIjogIjIwMjYtMDMtMzEiLAogICAgICAidGVzdF9zdGFydCI6ICIyMDI2LTA0LTA4IiwKICAgICAgInRlc3RfZW5kIjogIjIwMjYtMDktMzAiLAogICAgICAiZW1iYXJnb19kYXlzIjogNywKICAgICAgIm1heF90cmFpbl9vcmlnaW4iOiAiMjAyNi0wMy0yMyIsCiAgICAgICJyZWdpbWUiOiAiTU9ERVJOIgogICAgfQogIF0sCiAgInNlcmllc19lbGlnaWJpbGl0eSI6IHsKICAgICJ0aWVyX2EiOiB7CiAgICAgICJtaW5fdHJhaW5fb2JzZXJ2YXRpb25zIjogNjAsCiAgICAgICJtaW5fY292ZXJhZ2UiOiAwLjMwLAogICAgICAiY292ZXJhZ2Vfc3Bhbl9kZWZpbml0aW9uIjogImluY2x1c2l2ZSBmaXJzdC10by1sYXN0IG9ic2VydmVkIHRyYWRpbmcgZGF0ZSB3aXRoaW4gdHJhaW5pbmcgd2luZG93IiwKICAgICAgIm9yaWdpbl9yZWNlbmN5X3dpbmRvd19kYXlzIjogMzAsCiAgICAgICJtaW5fb3JpZ2luX3JlY2VuY3lfb2JzZXJ2YXRpb25zIjogMwogICAgfSwKICAgICJ0aWVyX2IiOiB7CiAgICAgICJtaW5fdHJhaW5fb2JzZXJ2YXRpb25zIjogMTIwLAogICAgICAibWluX2NvdmVyYWdlIjogMC4zMCwKICAgICAgImNvdmVyYWdlX3NwYW5fZGVmaW5pdGlvbiI6ICJpbmNsdXNpdmUgZmlyc3QtdG8tbGFzdCBvYnNlcnZlZCB0cmFkaW5nIGRhdGUgd2l0aGluIHRyYWluaW5nIHdpbmRvdyIsCiAgICAgICJvcmlnaW5fcmVjZW5jeV93aW5kb3dfZGF5cyI6IDMwLAogICAgICAibWluX29yaWdpbl9yZWNlbmN5X29ic2VydmF0aW9ucyI6IDMKICAgIH0KICB9LAogICJmZWF0dXJlcyI6IHsKICAgICJidWlsZGVyX21vZHVsZSI6ICJzcmMuZm9yZWNhc3RfZmVhdHVyZXMiLAogICAgImZlYXR1cmVfY29kZV92ZXJzaW9uIjogIjEuMC4wIiwKICAgICJmZWF0dXJlX2NvbHVtbnMiOiBbCiAgICAgICJjdXJyZW50X3ByaWNlIiwKICAgICAgInByaWNlX2xhZ18xIiwKICAgICAgInByaWNlX2xhZ18zIiwKICAgICAgInByaWNlX2xhZ183IiwKICAgICAgInByaWNlX2xhZ18xNCIsCiAgICAgICJwcmljZV9sYWdfMjgiLAogICAgICAibGFzdF9vYnNlcnZhdGlvbl9hZ2VfZGF5cyIsCiAgICAgICJyb2xsaW5nX2NvdW50XzdkIiwKICAgICAgInJvbGxpbmdfbWVhbl83ZCIsCiAgICAgICJyb2xsaW5nX3N0ZF83ZCIsCiAgICAgICJyb2xsaW5nX2NvdW50XzMwZCIsCiAgICAgICJyb2xsaW5nX21lYW5fMzBkIiwKICAgICAgInJvbGxpbmdfc3RkXzMwZCIsCiAgICAgICJkYXlfb2Zfd2VlayIsCiAgICAgICJkYXlfb2Zfd2Vla19zaW4iLAogICAgICAiZGF5X29mX3dlZWtfY29zIiwKICAgICAgIm1vbnRoIiwKICAgICAgIm1vbnRoX3NpbiIsCiAgICAgICJtb250aF9jb3MiLAogICAgICAiZGF5X29mX21vbnRoIiwKICAgICAgImRheV9vZl95ZWFyIgogICAgXSwKICAgICJjYWxlbmRhcl9sYWdzIjogWzEsIDMsIDcsIDE0LCAyOF0sCiAgICAicm9sbGluZ193aW5kb3dzX2RheXMiOiBbNywgMzBdLAogICAgInJvbGxpbmdfd2luZG93X2NvbnZlbnRpb24iOiAiUHJlY2VkaW5nIGNhbGVuZGFyIHNwYW4gW3QtVywgdC0xXSBzdHJpY3RseSBleGNsdXNpdmUgb2YgdCIsCiAgICAic3RkX2Rkb2YiOiAxLAogICAgIm1pbl9vYnNfZm9yX21lYW4iOiAxLAogICAgIm1pbl9vYnNfZm9yX3N0ZCI6IDIsCiAgICAiaW1wdXRhdGlvbl9wb2xpY3kiOiAiVHJhaW5pbmctb25seSBmaXR0ZWQgbWVkaWFuIGltcHV0ZXIgZm9yIG1vZGVscyByZXF1aXJpbmcgZGVuc2UgaW5wdXRzLiBOYXRpdmUgbWlzc2luZyB2YWx1ZSBoYW5kbGluZyBmb3IgSGlzdEdyYWRpZW50Qm9vc3RpbmcuIFplcm8gZnV0dXJlIG9yIHRlc3QgaW1wdXRhdGlvbiBsZWFrYWdlLiIKICB9LAogICJpbm5lcl92YWxpZGF0aW9uIjogewogICAgInNwbGl0c19wZXJfZm9sZCI6IDIsCiAgICAic3BsaXRfMSI6IHsKICAgICAgIm5hbWUiOiAiaW5uZXJfc3BsaXRfMSIsCiAgICAgICJ2YWxfb2Zmc2V0X3N0YXJ0X2RheXMiOiAxMTksCiAgICAgICJ2YWxfb2Zmc2V0X2VuZF9kYXlzIjogNjAsCiAgICAgICJlbWJhcmdvX2RheXMiOiA3CiAgICB9LAogICAgInNwbGl0XzIiOiB7CiAgICAgICJuYW1lIjogImlubmVyX3NwbGl0XzIiLAogICAgICAidmFsX29mZnNldF9zdGFydF9kYXlzIjogNTksCiAgICAgICJ2YWxfb2Zmc2V0X2VuZF9kYXlzIjogMCwKICAgICAgImVtYmFyZ29fZGF5cyI6IDcKICAgIH0sCiAgICAic2VsZWN0aW9uX21ldHJpYyI6ICJwb29sZWRfaW5uZXJfdmFsX21hZSIsCiAgICAiZGV0ZXJtaW5pc3RpY190aWVfYnJlYWtlciI6ICJzaW1wbGVyX21vZGVsX2hpZ2hlcl9yZWd1bGFyaXphdGlvbiIsCiAgICAiZmFsbGJhY2tfZGVmYXVsdHMiOiB7CiAgICAgICJyaWRnZSI6IDEuMCwKICAgICAgImhpc3RfZ3JhZGllbnRfYm9vc3RpbmciOiAiaGdiX2RlcHRoMyIKICAgIH0KICB9LAogICJtaXNzaW5nX2RhdGFfdGF4b25vbXkiOiB7CiAgICAiTk9fT1JJR0lOIjogIlNlcmllcyB3YXMgbm90IG9ic2VydmVkIG9uIGNhbmRpZGF0ZSBjYWxlbmRhciBvcmlnaW4gZGF0ZSB0IiwKICAgICJJTkVMSUdJQkxFX0hJU1RPUlkiOiAiU2VyaWVzIGZhaWxlZCBUaWVyIEEgKE4gPCA2MCBvciBjb3ZlcmFnZSA8IDAuMzApIG9yIFRpZXIgQiAoTiA8IDEyMCBvciBjb3ZlcmFnZSA8IDAuMzApIHRyYWluaW5nIHRocmVzaG9sZCIsCiAgICAiTUlTU0lOR19GRUFUVVJFUyI6ICJPcmlnaW4gbGFja3MgcmVxdWlyZWQgZmVhdHVyZSBoaXN0b3J5IChlLmcuIGxhc3Rfb2JzZXJ2YXRpb25fYWdlX2RheXMgaXMgTmFOKSIsCiAgICAiTk9fVEFSR0VUIjogIk5vIHZhbGlkIHRyYWRlIG9ic2VydmF0aW9uIGV4aXN0cyBpbiB0YXJnZXQgZXZhbHVhdGlvbiB3aW5kb3cgW3QrNiwgdCs4XSAoZXZhbHVhdGlvbiBvcmlnaW4gc2tpcHBlZCkiLAogICAgIk1PREVMX0FCU1RFTlRJT04iOiAiTW9kZWwgYWJzdGFpbnMgZnJvbSBwcmVkaWN0aW5nIGR1ZSB0byB1bmNlcnRhaW50eSwgb3V0LW9mLWRvbWFpbiBmZWF0dXJlcywgb3Igbm9uLWZpbml0ZSBvdXRwdXRzIiwKICAgICJOT05fUE9TSVRJVkVfUkVDT05TVFJVQ1RFRF9GT1JFQ0FTVCI6ICJSZWNvbnN0cnVjdGVkIGZvcmVjYXN0IHlfaGF0ID0geV90ICsgZGVsdGEgaXMgbm9uLXBvc2l0aXZlIG9yIG5vbi1maW5pdGUiLAogICAgIlNFUlZJTkdfREVOSUFMIjogIlNlcnZpbmcgcG9saWN5IGRlbmlhbDogbWFya2V0IGFsaWFzIHVuZGVyIHJldmlldywgdW5rbm93biBncmFkZSwgb3Igb2JzZXJ2YXRpb24gb2xkZXIgdGhhbiAzIGNhbGVuZGFyIGRheXMiCiAgfSwKICAiY2FuZGlkYXRlX21vZGVscyI6IHsKICAgICJyaWRnZV9yZWdyZXNzaW9uIjogewogICAgICAibW9kZWxfY2xhc3MiOiAic2tsZWFybi5saW5lYXJfbW9kZWwuUmlkZ2UiLAogICAgICAicm9sZSI6ICJMaW5lYXIgcmVndWxhcml6ZWQgYmFzZWxpbmUgb24gbGFnZ2VkIHByaWNlIGFuZCBjYWxlbmRhciBmZWF0dXJlcyIsCiAgICAgICJoeXBlcnBhcmFtZXRlcnMiOiB7CiAgICAgICAgImFscGhhIjogWzAuMSwgMS4wLCAxMC4wXQogICAgICB9LAogICAgICAicHJlcHJvY2Vzc29yIjogIlBpcGVsaW5lKFsoJ2ltcHV0ZXInLCBTaW1wbGVJbXB1dGVyKHN0cmF0ZWd5PSdtZWRpYW4nKSksICgnc2NhbGVyJywgU3RhbmRhcmRTY2FsZXIoKSldKSBmaXR0ZWQgc3RyaWN0bHkgb24gdHJhaW4gZm9sZCIsCiAgICAgICJ0dW5pbmdfcHJvdG9jb2wiOiAiSW5uZXIgY2hyb25vbG9naWNhbCByb2xsaW5nLW9yaWdpbiB2YWxpZGF0aW9uIG9uIHRyYWluIGZvbGQ7IHplcm8gdGVzdCBmb2xkIGluc3BlY3Rpb24iCiAgICB9LAogICAgImhpc3RfZ3JhZGllbnRfYm9vc3RpbmciOiB7CiAgICAgICJtb2RlbF9jbGFzcyI6ICJza2xlYXJuLmVuc2VtYmxlLkhpc3RHcmFkaWVudEJvb3N0aW5nUmVncmVzc29yIiwKICAgICAgInJvbGUiOiAiTm9ubGluZWFyIHRhYnVsYXIgdHJlZSBiYXNlbGluZSB3aXRoIG5hdGl2ZSBOYU4gc3VwcG9ydCIsCiAgICAgICJjb25maWdzIjogWwogICAgICAgIHsKICAgICAgICAgICJjb25maWdfaWQiOiAiaGdiX2RlcHRoMyIsCiAgICAgICAgICAibWF4X2l0ZXIiOiAxMDAsCiAgICAgICAgICAibWF4X2RlcHRoIjogMywKICAgICAgICAgICJsZWFybmluZ19yYXRlIjogMC4wNSwKICAgICAgICAgICJtaW5fc2FtcGxlc19sZWFmIjogMjAsCiAgICAgICAgICAiZWFybHlfc3RvcHBpbmciOiBmYWxzZSwKICAgICAgICAgICJyYW5kb21fc3RhdGUiOiA0MgogICAgICAgIH0sCiAgICAgICAgewogICAgICAgICAgImNvbmZpZ19pZCI6ICJoZ2JfZGVwdGg1IiwKICAgICAgICAgICJtYXhfaXRlciI6IDEwMCwKICAgICAgICAgICJtYXhfZGVwdGgiOiA1LAogICAgICAgICAgImxlYXJuaW5nX3JhdGUiOiAwLjA1LAogICAgICAgICAgIm1pbl9zYW1wbGVzX2xlYWYiOiAzMCwKICAgICAgICAgICJlYXJseV9zdG9wcGluZyI6IGZhbHNlLAogICAgICAgICAgInJhbmRvbV9zdGF0ZSI6IDQyCiAgICAgICAgfQogICAgICBdLAogICAgICAicHJlcHJvY2Vzc29yIjogIk5vbmUgKG5hdGl2ZSBOYU4gc3VwcG9ydCkiLAogICAgICAidHVuaW5nX3Byb3RvY29sIjogIklubmVyIGNocm9ub2xvZ2ljYWwgcm9sbGluZy1vcmlnaW4gdmFsaWRhdGlvbiBvbiB0cmFpbiBmb2xkOyB6ZXJvIHRlc3QgZm9sZCBpbnNwZWN0aW9uIgogICAgfSwKICAgICJiZW5jaG1hcmtfYmFzZWxpbmVzX3Jlc2NvcmVkIjogWwogICAgICB7CiAgICAgICAgIm5hbWUiOiAibmFpdmUiLAogICAgICAgICJmb3JtdWxhIjogInlfaGF0ID0geSh0KSIsCiAgICAgICAgInNvdXJjZSI6ICJzcmMuYmFzZWxpbmVzLnByZWRpY3RfbmFpdmUiCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAibmFtZSI6ICJtb3ZpbmdfYXZlcmFnZV83b2JzX2luXzMwZCIsCiAgICAgICAgImZvcm11bGEiOiAiTWVhbiBvZiBsYXN0IDw9NyBvYnNlcnZhdGlvbnMgd2l0aGluIFt0LTMwZCwgdF0sIG1pbiAzIG9ic2VydmF0aW9ucyIsCiAgICAgICAgInNvdXJjZSI6ICJzcmMuYmFzZWxpbmVzLnByZWRpY3RfbW92aW5nX2F2ZXJhZ2UiLAogICAgICAgICJub3RlIjogIkRpc3Rpbmd1aXNoZWQgZnJvbSBmZWF0dXJlIHJvbGxpbmcgNyBjYWxlbmRhciBkYXlzIgogICAgICB9LAogICAgICB7CiAgICAgICAgIm5hbWUiOiAic2Vhc29uYWxfbmFpdmVfMzU3ZCIsCiAgICAgICAgImZvcm11bGEiOiAiT2JzZXJ2YXRpb24gbmVhcmVzdCB0byAodCArIDcgLSAzNjQpID0gKHQgLSAzNTcpIHdpdGhpbiArLy0zIGRheXMsIGVhcmxpZXIgdGllLWJyZWFrIiwKICAgICAgICAic291cmNlIjogInNyYy5iYXNlbGluZXMucHJlZGljdF9zZWFzb25hbF9uYWl2ZSIKICAgICAgfSwKICAgICAgewogICAgICAgICJuYW1lIjogImRyaWZ0IiwKICAgICAgICAiZm9ybXVsYSI6ICJ5KHQpICsgMC41ICogc2xvcGUgKiA3LCBhbmNob3IgbmVhcmVzdCAodCAtIDE0KSB3aXRoaW4gKy8tMyBkYXlzIiwKICAgICAgICAic291cmNlIjogInNyYy5iYXNlbGluZXMucHJlZGljdF9kcmlmdCIKICAgICAgfQogICAgXQogIH0sCiAgImV2YWx1YXRpb25fbWV0cmljcyI6IHsKICAgICJwb2ludF9tZXRyaWNzIjogWwogICAgICAiTUFFIiwKICAgICAgIlJNU0UiLAogICAgICAiTWVhbkJpYXMiLAogICAgICAiTWVkaWFuQUUiLAogICAgICAiTUFQRV93aXRoX3ZhbGlkX2Rlbm9taW5hdG9yIiwKICAgICAgInBjdF93aXRoaW5fNXBjdCIKICAgIF0sCiAgICAicGFpcmVkX3NraWxsIjogewogICAgICAiZm9ybXVsYSI6ICIxIC0gKHN1bSh8eSAtIHlfaGF0fCkgLyBzdW0ofHkgLSB5X25haXZlfCkpIG9uIGV4YWN0bHkgbWF0Y2hlZCBub24tYWJzdGFpbmVkIGZpbml0ZSBwYWlycyIsCiAgICAgICJjb25maWRlbmNlX2ludGVydmFscyI6ICJDbHVzdGVyIGJvb3RzdHJhcCBhdCBzZXJpZXMgbGV2ZWwgKDEwMDAgcmVzYW1wbGVzLCA5NSUgQ0ksIHNlZWQgNDIpIgogICAgfSwKICAgICJzbGljZV9zdHJhdGlmaWNhdGlvbiI6IFsKICAgICAgIk92ZXJhbGwgVGllciBBIiwKICAgICAgIk92ZXJhbGwgVGllciBCIiwKICAgICAgIkJ5IERpc3RyaWN0IChHaGF6aWFiYWQsIEpoYW5zaSwgS2FucHVyLCBMdWNrbm93LCBVbm5hbykiLAogICAgICAiQnkgRm9jdXMgQ3JvcCAoV2hlYXQsIFBvdGF0bywgT25pb24sIFRvbWF0bywgUGFkZHkoRGhhbikoQ29tbW9uKSwgUmljZSkiLAogICAgICAiQnkgUmVnaW1lIChISVNUT1JJQ0FMLCBUUkFOU0lUSU9OLCBNT0RFUk4pIgogICAgXSwKICAgICJzZW5zaXRpdml0aWVzIjogWwogICAgICAiZXhhY3RfdDciLAogICAgICAiZGlzcHV0ZWRfNzdfa2V5cyIsCiAgICAgICJyZXRyb3NwZWN0aXZlX2d0XzJwY3RfbW92ZSIKICAgIF0sCiAgICAibWFzZV9wb2xpY3kiOiAiTk9UIENPTVBVVEVEIHVudGlsIG9mZmljaWFsIHRyYWluLW9ubHkgbm9uc2Vhc29uYWwgMS1zdGVwIHNjYWxpbmcgaW1wbGVtZW50YXRpb24gaXMgdmVyaWZpZWQiCiAgfQp9Cg==",
    "size_bytes": 9310,
    "sha256": "cd875d249a126839fa4e035550d8ea51a965f9ffac3ac9bcd4d31f35d88ee938"
  }
}

print("\n=== DECODING & VERIFYING PAYLOADS ===")
target_paths = {
    "market_identity.py": src_dir / "market_identity.py",
    "data_pipeline.py": src_dir / "data_pipeline.py",
    "baselines.py": src_dir / "baselines.py",
    "forecast_features.py": src_dir / "forecast_features.py",
    "forecast_benchmark.py": src_dir / "forecast_benchmark.py",
    "market_aliases.csv": ref_dir / "market_aliases.csv",
    "price_forecast_v1.json": cfg_dir / "price_forecast_v1.json",
}

for fname, meta in payload_specs.items():
    dest = target_paths[fname]
    raw_bytes = base64.b64decode(meta["content_b64"].encode("ascii"))
    actual_sha = hashlib.sha256(raw_bytes).hexdigest()
    expected_sha = meta["sha256"]
    if actual_sha != expected_sha:
        raise ValueError(f"Payload integrity failure for {fname}! Expected {expected_sha}, got {actual_sha}")
    dest.write_bytes(raw_bytes)
    print(f"Decoded {fname}: {len(raw_bytes):,} bytes, SHA-256 confirmed ({actual_sha[:16]}...)")

# Locate Kaggle raw input dataset
KAGGLE_INPUT_DIR = "/kaggle/input/mandi-prediction-audit-003b"
if not os.path.exists(KAGGLE_INPUT_DIR):
    KAGGLE_INPUT_DIR = "/kaggle/input/datasets/theankitchoubey/mandi-prediction-audit-003b"
if not os.path.exists(KAGGLE_INPUT_DIR):
    cands = glob.glob("/kaggle/input/**/mandi_5year_validated.csv", recursive=True)
    if cands:
        KAGGLE_INPUT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(cands[0])))
    else:
        KAGGLE_INPUT_DIR = "."

print(f"\nResolved Dataset Payload Root: {KAGGLE_INPUT_DIR}")

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

raw_csv_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "5 year dataset/*.csv")))
if not raw_csv_files:
    raw_csv_files = sorted(glob.glob("5 year dataset/*.csv"))
print(f"Found {len(raw_csv_files)} raw CSV files.")
for f in raw_csv_files:
    print(f"  RAW: {os.path.basename(f)} ({os.path.getsize(f):,} bytes)")


In [ ]:
# Cell 2: Pipeline v2 Execution and Panel v2 SHA-256 Integrity Assertion Gate
import os
import sys
import glob
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd

from src.data_pipeline import MandiDataPipeline

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
ref_dir = working_root / "data" / "reference"

print("=== EXECUTING PIPELINE V2 REGENERATION ===")
df_alias_all = pd.read_csv(ref_dir / "market_aliases.csv")
df_alias_accepted = df_alias_all[df_alias_all["status"] == "ACCEPTED"].copy()
print(f"Loaded {len(df_alias_accepted)} accepted alias entries.")

pipeline_v2 = MandiDataPipeline(
    input_dir=os.path.join(KAGGLE_INPUT_DIR, "5 year dataset"),
    output_dir=".",
    output_filename="mandi_5year_validated_v2.csv",
    manifest_filename="processing_manifest_v2.json",
    min_series_records=20,
    exclude_ambiguous_same_grade=True,
    pipeline_version="v2",
    alias_df=df_alias_accepted
)

df_v2, manifest_v2, out_csv_v2, out_manifest_v2 = pipeline_v2.run(raw_csv_files)
print(f"Pipeline v2 Complete! Clean rows: {len(df_v2):,}, unique series: {df_v2['series_id'].nunique():,}")

# CRITICAL SHA-256 INTEGRITY GATE ASSERTION
v2_sha256 = compute_sha256("mandi_5year_validated_v2.csv")
print(f"\nPanel v2 SHA-256: {v2_sha256}")
expected_sha256 = "851e718468d6b8c9fd07944535a462338e71aa105af25339fe9438b697088f31"
if v2_sha256 != expected_sha256:
    raise RuntimeError(
        f"PROVISIONAL PANEL V2 SHA-256 MISMATCH! Got {v2_sha256}, expected {expected_sha256}. STOPPING EXECUTION."
    )
print(">>> PANEL V2 INTEGRITY ASSERTION PASSED SUCCESSFULLY! SHA-256 CONFIRMED. <<<")

# Check Row Conservation: 472385 = 463217 + 855 + 2936 + 5350 + 27
summary = manifest_v2.get("summary_metrics", manifest_v2)
raw_total = summary.get("total_raw_rows", 0)
acc_records = summary.get("final_clean_rows", 0)
exact_dup = summary.get("exact_duplicates_deduplicated", 0)
quar_conflict = summary.get("ambiguous_same_grade_excluded_rows", 0)
quar_depth = summary.get("low_observation_series_dropped_rows", 0)
quar_invalid = summary.get("total_dropped_raw_rows", 0)

sum_partition = acc_records + exact_dup + quar_conflict + quar_depth + quar_invalid
print(f"Row Conservation: {raw_total} = {acc_records} + {exact_dup} + {quar_conflict} + {quar_depth} + {quar_invalid}")
if sum_partition != raw_total or raw_total != 472385:
    raise RuntimeError(f"ROW CONSERVATION VIOLATION! Expected 472,385, got {raw_total} (sum {sum_partition}). STOPPING.")
print(">>> ROW CONSERVATION CONFIRMED: 472,385 = 463,217 + 855 + 2,936 + 5,350 + 27 <<<")

# Assert finite positive prices
prices = df_v2["Modal_Price"].values
if not np.all(np.isfinite(prices)) or not np.all(prices > 0):
    raise ValueError("Panel v2 contains non-finite or non-positive prices!")
print(f"Price Integrity: all {len(prices):,} modal prices are strictly positive and finite.")


In [ ]:
# Cell 3: Tiny Guarded Model-Fit, Serialization & Response Smoke Test
import os
import sys
import math
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from src.forecast_benchmark import (
    build_candidate_ridge_model,
    build_candidate_hgb_model,
    FEATURE_COLUMNS,
)

print("=== EXECUTING TINY GUARDED SMOKE TEST ===")
working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
# 1. Create small synthetic feature matrix
rng = np.random.default_rng(42)
X_smoke = pd.DataFrame(rng.normal(size=(10, len(FEATURE_COLUMNS))), columns=FEATURE_COLUMNS)
X_smoke.loc[0, "price_lag_1"] = np.nan  # test missing value handling
y_delta_smoke = rng.normal(loc=5.0, scale=10.0, size=10)

# 2. Fit Ridge
ridge_smoke = build_candidate_ridge_model(alpha=1.0)
ridge_smoke.fit(X_smoke, y_delta_smoke)
ridge_pred_orig = ridge_smoke.predict(X_smoke)

# 3. Fit HGB
hgb_smoke = build_candidate_hgb_model(max_depth=3, min_samples_leaf=2)
hgb_smoke.fit(X_smoke, y_delta_smoke)
hgb_pred_orig = hgb_smoke.predict(X_smoke)

# 4. Serialize and reload
smoke_ridge_path = working_root / "smoke_ridge.joblib"
smoke_hgb_path = working_root / "smoke_hgb.joblib"
joblib.dump(ridge_smoke, smoke_ridge_path)
joblib.dump(hgb_smoke, smoke_hgb_path)

ridge_loaded = joblib.load(smoke_ridge_path)
hgb_loaded = joblib.load(smoke_hgb_path)

ridge_pred_loaded = ridge_loaded.predict(X_smoke)
hgb_pred_loaded = hgb_loaded.predict(X_smoke)

assert np.allclose(ridge_pred_orig, ridge_pred_loaded), "Ridge serialization mismatch!"
assert np.allclose(hgb_pred_orig, hgb_pred_loaded), "HGB serialization mismatch!"
print("Smoke test: serialization and reload equality confirmed (np.allclose = True).")

# 5. Response transformation & abstention check
y_t_dummy = np.array([100.0, 50.0, 10.0, 5.0, -100.0, 20.0, 30.0, 40.0, 50.0, 60.0])
delta_dummy = np.array([10.0, -20.0, -15.0, -10.0, 20.0, 5.0, 0.0, -5.0, 10.0, 15.0])
reconstructed = y_t_dummy + delta_dummy
# Cases with reconstructed <= 0 or not finite must abstain
abstentions = []
for val in reconstructed:
    if not math.isfinite(val) or val <= 0:
        abstentions.append("NON_POSITIVE_RECONSTRUCTED_FORECAST")
    else:
        abstentions.append("SCORED")

assert abstentions[2] == "NON_POSITIVE_RECONSTRUCTED_FORECAST"  # 10 - 15 = -5
assert abstentions[3] == "NON_POSITIVE_RECONSTRUCTED_FORECAST"  # 5 - 10 = -5
assert abstentions[4] == "NON_POSITIVE_RECONSTRUCTED_FORECAST"  # -100 + 20 = -80
assert abstentions[0] == "SCORED"  # 100 + 10 = 110
print("Smoke test: response transformation & non-positive abstention logic verified.")

# Clean up smoke files
if smoke_ridge_path.exists():
    smoke_ridge_path.unlink()
if smoke_hgb_path.exists():
    smoke_hgb_path.unlink()
print(">>> TINY GUARDED SMOKE TEST COMPLETED SUCCESSFULLY. PROCEEDING TO FULL WORKLOAD. <<<")


In [ ]:
# Cell 4: End-to-End Benchmark Execution across Folds F1-F4
import os
import sys
import time
import datetime
import math
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from src.forecast_benchmark import (
    ForecastBenchmarkEngine,
    SeriesIdentity,
    get_standard_folds,
    FEATURE_COLUMNS,
)

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
models_dir = working_root / "models"
outputs_fc007_dir = working_root / "experiments" / "outputs" / "FC007"
models_dir.mkdir(parents=True, exist_ok=True)
outputs_fc007_dir.mkdir(parents=True, exist_ok=True)

print("=== INITIALIZING FORECAST BENCHMARK ENGINE ===")
engine = ForecastBenchmarkEngine()

# Group panel v2 records into clean series history map
print("Grouping panel v2 into clean series history map...")
records_v2 = df_v2.to_dict(orient="records")
clean_series_map, resolve_counts = engine.builder.resolve_raw_series_records(records_v2)
print(f"Resolved {len(clean_series_map):,} unique series identities.")
print(f"Accepted: {resolve_counts.get('accepted', 0):,}, Conflicts: {resolve_counts.get('conflict', 0):,}, Duplicates: {resolve_counts.get('duplicate', 0):,}")

all_fold_preds = []
inner_tuning_manifest = {}
outer_model_manifest = {}

for fold in engine.folds:
    print(f"\n=======================================================")
    print(f"EXECUTING FOLD: {fold.fold_id} ({fold.regime})")
    print(f"Train cutoff: {fold.train_cutoff}, Max train origin: {fold.max_train_origin}")
    print(f"Test window: {fold.test_start} .. {fold.test_end} (Embargo: {fold.embargo_days}d)")
    print(f"=======================================================")

    # Step 4.1: Inner Validation Tuning
    t0_tune = time.time()
    print("Running inner chronological validation tuning...")
    tuning_res = engine.tune_candidate_models(clean_series_map, fold)
    tune_time = time.time() - t0_tune

    best_alpha = tuning_res["best_ridge_alpha"]
    best_hgb = tuning_res["best_hgb_config"]
    print(f"Inner tuning completed in {tune_time:.1f}s.")
    print(f"  Selected Ridge alpha: {best_alpha}")
    print(f"  Selected HGB config: {best_hgb.get('config_id')} (max_depth={best_hgb.get('max_depth')}, lr={best_hgb.get('learning_rate')})")
    inner_tuning_manifest[fold.fold_id] = tuning_res

    # Step 4.2: Prepare Outer Dataset Partitions
    t0_part = time.time()
    print("Preparing outer dataset partitions (train & test)...")
    train_df, test_df = engine.prepare_dataset_partitions(clean_series_map, fold)
    part_time = time.time() - t0_part
    print(f"Partitions prepared in {part_time:.1f}s.")
    print(f"  Train samples: {len(train_df):,} rows (TierA: {sum(train_df['tier'] == 'TierA'):,}, TierB: {sum(train_df['tier'] == 'TierB'):,})")
    print(f"  Test samples: {len(test_df):,} rows (TierA: {sum(test_df['tier'] == 'TierA'):,}, TierB: {sum(test_df['tier'] == 'TierB'):,})")

    # Assert Label Maturity & Temporal Bounds
    assert all(train_df["target_date"] <= fold.train_cutoff), f"Training label maturity leak in fold {fold.fold_id}!"
    assert all(train_df["origin_date"] <= fold.max_train_origin), f"Training origin leak in fold {fold.fold_id}!"
    assert all(fold.test_start <= d <= fold.test_end for d in test_df["origin_date"]), f"Test origin out of bounds in fold {fold.fold_id}!"
    assert all(d <= fold.test_end for d in test_df["target_date"]), f"Test target out of bounds in fold {fold.fold_id}!"
    print("  Temporal boundaries and label maturity assertions verified.")

    # Step 4.3: Refit Candidate Models on Outer Train & Predict Outer Test
    t0_fit = time.time()
    print("Refitting selected candidate models on full outer training partition...")
    pred_df, models_dict = engine.train_and_predict_outer(
        train_df,
        test_df,
        ridge_alpha=best_alpha,
        hgb_config=best_hgb,
    )
    fit_time = time.time() - t0_fit
    print(f"Refit and inference completed in {fit_time:.1f}s.")

    # Check abstentions
    ridge_abst = (pred_df["ridge_abstention"] == "NON_POSITIVE_RECONSTRUCTED_FORECAST").sum()
    hgb_abst = (pred_df["hgb_abstention"] == "NON_POSITIVE_RECONSTRUCTED_FORECAST").sum()
    print(f"  Ridge non-positive abstentions: {ridge_abst:,} / {len(pred_df):,}")
    print(f"  HGB non-positive abstentions: {hgb_abst:,} / {len(pred_df):,}")

    # Step 4.4: Save fitted models and manifests remotely
    ridge_model_path = models_dir / f"ridge_fold_{fold.fold_id}.joblib"
    hgb_model_path = models_dir / f"hgb_fold_{fold.fold_id}.joblib"
    joblib.dump(models_dict["ridge_model"], ridge_model_path)
    joblib.dump(models_dict["hgb_model"], hgb_model_path)

    outer_model_manifest[fold.fold_id] = {
        "fold_id": fold.fold_id,
        "regime": fold.regime,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "ridge_alpha": best_alpha,
        "ridge_model_file": str(ridge_model_path.name),
        "ridge_model_sha256": compute_sha256(ridge_model_path),
        "hgb_config": best_hgb,
        "hgb_model_file": str(hgb_model_path.name),
        "hgb_model_sha256": compute_sha256(hgb_model_path),
        "feature_columns": FEATURE_COLUMNS,
    }

    all_fold_preds.append(pred_df)

# Concatenate all fold predictions
master_pred_df = pd.concat(all_fold_preds, ignore_index=True)
print(f"\n=== ALL FOLDS COMPLETE ===")
print(f"Total Master Outer Predictions: {len(master_pred_df):,} rows.")

# Export full prediction archive remotely
full_pred_parquet = outputs_fc007_dir / "outer_predictions_fc007.parquet"
full_pred_csv_gz = outputs_fc007_dir / "outer_predictions_fc007.csv.gz"
master_pred_df.to_parquet(full_pred_parquet, index=False)
master_pred_df.to_csv(full_pred_csv_gz, index=False, compression="gzip")

parquet_sha = compute_sha256(full_pred_parquet)
csvgz_sha = compute_sha256(full_pred_csv_gz)
print(f"Exported full prediction archive:")
print(f"  Parquet: {full_pred_parquet} ({os.path.getsize(full_pred_parquet):,} bytes, SHA: {parquet_sha[:16]}...)")
print(f"  CSV.GZ:  {full_pred_csv_gz} ({os.path.getsize(full_pred_csv_gz):,} bytes, SHA: {csvgz_sha[:16]}...)")


In [ ]:
# Cell 5: Evaluation Scorecards, Paired Naive Skill & Series-Cluster Bootstrap 95% CIs
import os
import sys
import math
from pathlib import Path
import numpy as np
import pandas as pd

from src.forecast_benchmark import compute_slice_metrics

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
ref_dir = working_root / "data" / "reference"
reports_fc007_dir = working_root / "experiments" / "reports" / "FC007"
reports_fc007_dir.mkdir(parents=True, exist_ok=True)

print("=== COMPUTING EVALUATION SCORECARDS & BOOTSTRAP CIs ===")

# Define model columns
model_cols = {
    "naive": "naive_pred",
    "moving_average_7obs_in_30d": "ma_pred",
    "seasonal_naive_357d": "seasonal_pred",
    "drift": "drift_pred",
    "ridge": "ridge_pred",
    "hist_gradient_boosting": "hgb_pred",
}

focus_crops = [
    "Wheat",
    "Potato",
    "Onion",
    "Tomato",
    "Paddy(Dhan)(Common)",
    "Rice",
]

target_districts = [
    "Ghaziabad",
    "Jhansi",
    "Kanpur",
    "Lucknow",
    "Unnao",
]

# Identify disputed 77 canonical market alias names
df_alias = pd.read_csv(ref_dir / "market_aliases.csv")
canonical_markets = set(df_alias[df_alias["status"] == "ACCEPTED"]["market_raw"].dropna().unique())

# Retrospective > 2% target move diagnostic column
master_pred_df["gt_2pct_move"] = (
    (master_pred_df["y_true"] - master_pred_df["y_t"]).abs() / master_pred_df["y_t"] > 0.02
)
master_pred_df["is_disputed_market"] = master_pred_df["market"].isin(canonical_markets)

def run_scorecard_for_slices(slice_dict, slice_type):
    rows = []
    for slice_name, sub_df in slice_dict.items():
        if len(sub_df) == 0:
            continue
        for m_name, p_col in model_cols.items():
            res = compute_slice_metrics(
                sub_df,
                pred_col=p_col,
                true_col="y_true",
                naive_col="naive_pred",
                n_bootstrap=1000,
                seed=42,
            )
            r = {
                "slice_type": slice_type,
                "slice_name": slice_name,
                "model": m_name,
                "total_samples": res["total_samples"],
                "scored_count": res["scored_count"],
                "coverage": round(res["coverage"], 4),
                "unique_series": res["unique_series"],
                "is_low_n": res["is_low_n"],
                "mae": round(res["mae"], 2) if not np.isnan(res["mae"]) else np.nan,
                "mae_ci_95": f"[{res['mae_ci_95'][0]:.2f}, {res['mae_ci_95'][1]:.2f}]" if not np.isnan(res["mae_ci_95"][0]) else "NA",
                "rmse": round(res["rmse"], 2) if not np.isnan(res["rmse"]) else np.nan,
                "bias": round(res["bias"], 2) if not np.isnan(res["bias"]) else np.nan,
                "median_ae": round(res["median_ae"], 2) if not np.isnan(res["median_ae"]) else np.nan,
                "mape": round(res["mape"], 2) if not np.isnan(res["mape"]) else np.nan,
                "pct_within_5pct": round(res["pct_within_5pct"], 2) if not np.isnan(res["pct_within_5pct"]) else np.nan,
                "paired_naive_mae": round(res["paired_naive_mae"], 2) if not np.isnan(res["paired_naive_mae"]) else np.nan,
                "skill": round(res["skill"], 4) if not np.isnan(res["skill"]) else np.nan,
                "skill_ci_95": f"[{res['skill_ci_95'][0]:.4f}, {res['skill_ci_95'][1]:.4f}]" if not np.isnan(res['skill_ci_95'][0]) else "NA",
            }
            rows.append(r)
    return pd.DataFrame(rows)

# 1. Overall Scorecard (Combined, Tier A, Tier B)
overall_slices = {
    "Overall_Combined": master_pred_df,
    "Overall_TierA": master_pred_df[master_pred_df["tier"] == "TierA"],
    "Overall_TierB": master_pred_df[master_pred_df["tier"] == "TierB"],
}
df_scorecard_overall = run_scorecard_for_slices(overall_slices, "OVERALL")

# 2. By Fold Scorecard
fold_slices = {f"Fold_{f.fold_id}": master_pred_df[master_pred_df["fold"] == f.fold_id] for f in engine.folds}
df_scorecard_by_fold = run_scorecard_for_slices(fold_slices, "BY_FOLD")

# 3. By Crop Scorecard
crop_slices = {crop: master_pred_df[master_pred_df["commodity"] == crop] for crop in focus_crops}
df_scorecard_by_crop = run_scorecard_for_slices(crop_slices, "BY_CROP")

# 4. By Regime Scorecard (HISTORICAL: F1, TRANSITION: F2/F3, MODERN: F4)
regime_slices = {
    "HISTORICAL (F1)": master_pred_df[master_pred_df["fold"] == "F1"],
    "TRANSITION (F2, F3)": master_pred_df[master_pred_df["fold"].isin(["F2", "F3"])],
    "MODERN (F4)": master_pred_df[master_pred_df["fold"] == "F4"],
}
df_scorecard_by_regime = run_scorecard_for_slices(regime_slices, "BY_REGIME")

# 5. Sensitivities Scorecard
sensitivities_slices = {
    "Exact_T7_Subset": master_pred_df[master_pred_df["exact_t7"] == True],
    "Disputed_77_Keys_Subset": master_pred_df[master_pred_df["is_disputed_market"] == True],
    "Retrospective_GT_2pct_Move": master_pred_df[master_pred_df["gt_2pct_move"] == True],
}
df_scorecard_sensitivities = run_scorecard_for_slices(sensitivities_slices, "SENSITIVITIES")

# Export CSV scorecards under experiments/reports/FC007/
df_scorecard_overall.to_csv(reports_fc007_dir / "benchmark_scorecard_overall.csv", index=False)
df_scorecard_by_fold.to_csv(reports_fc007_dir / "benchmark_scorecard_by_fold.csv", index=False)
df_scorecard_by_crop.to_csv(reports_fc007_dir / "benchmark_scorecard_by_crop.csv", index=False)
df_scorecard_by_regime.to_csv(reports_fc007_dir / "benchmark_scorecard_by_regime.csv", index=False)
df_scorecard_sensitivities.to_csv(reports_fc007_dir / "benchmark_scorecard_sensitivities.csv", index=False)

print("Generated and exported 5 scorecard CSVs:")
for p in reports_fc007_dir.glob("*.csv"):
    print(f"  {p.name}: {p.stat().st_size:,} bytes")

print("\n=== HEADLINE OVERALL RESULTS ===")
print(df_scorecard_overall[["slice_name", "model", "scored_count", "mae", "mae_ci_95", "paired_naive_mae", "skill", "skill_ci_95"]].to_string(index=False))


In [ ]:
# Cell 6: In-Kernel Reconciliation Receipt, Output Manifest & Completion Assertion
import os
import sys
import json
import hashlib
import datetime
from pathlib import Path
import numpy as np
import pandas as pd

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
reports_fc007_dir = working_root / "experiments" / "reports" / "FC007"
outputs_fc007_dir = working_root / "experiments" / "outputs" / "FC007"
models_dir = working_root / "models"

print("=== EXECUTING IN-KERNEL RECONCILIATION ASSERTIONS ===")

# 1. Reconcile total predictions against fold sum
total_master_rows = len(master_pred_df)
fold_sum = sum(len(f_df) for f_df in all_fold_preds)
if total_master_rows != fold_sum:
    raise AssertionError(f"Reconciliation error: master rows ({total_master_rows}) != fold sum ({fold_sum})")
print(f"Assertion 1: Master predictions row count matches fold sum: {total_master_rows:,} == {fold_sum:,} (OK)")

# 2. Reconcile overall scorecard scored count
overall_scored_naive = df_scorecard_overall[
    (df_scorecard_overall["slice_name"] == "Overall_Combined") & (df_scorecard_overall["model"] == "naive")
]["scored_count"].iloc[0]
actual_valid_naive = int((master_pred_df["naive_pred"].notna() & master_pred_df["y_true"].notna()).sum())
if overall_scored_naive != actual_valid_naive:
    raise AssertionError(f"Reconciliation error: overall naive scored ({overall_scored_naive}) != actual valid ({actual_valid_naive})")
print(f"Assertion 2: Overall scorecard scored count matches raw valid mask: {overall_scored_naive:,} == {actual_valid_naive:,} (OK)")

# 3. Reconcile paired skill arithmetic
for model_name, col_name in [("ridge", "ridge_pred"), ("hist_gradient_boosting", "hgb_pred")]:
    sub = master_pred_df[master_pred_df[col_name].notna() & master_pred_df["naive_pred"].notna() & master_pred_df["y_true"].notna()]
    if len(sub) > 0:
        err_m = np.abs(sub["y_true"] - sub[col_name]).sum()
        err_n = np.abs(sub["y_true"] - sub["naive_pred"]).sum()
        manual_skill = 1.0 - (err_m / err_n)
        row_skill = df_scorecard_overall[
            (df_scorecard_overall["slice_name"] == "Overall_Combined") & (df_scorecard_overall["model"] == model_name)
        ]["skill"].iloc[0]
        assert abs(manual_skill - row_skill) < 1e-4, f"Skill arithmetic mismatch for {model_name}!"
        print(f"Assertion 3: Paired skill arithmetic verified for {model_name}: manual {manual_skill:.4f} == scorecard {row_skill:.4f} (OK)")

# 4. Save Inner Tuning Summary JSON
inner_tuning_summary_path = reports_fc007_dir / "inner_validation_tuning_summary.json"
# Clean tuning res to serializable types
serializable_tuning = {}
for fid, res in inner_tuning_manifest.items():
    serializable_tuning[fid] = {
        "fold_id": fid,
        "split_diagnostics": res.get("split_diagnostics"),
        "best_ridge_alpha": res.get("best_ridge_alpha"),
        "ridge_tuning_scores": res.get("ridge_tuning_scores"),
        "best_hgb_config": res.get("best_hgb_config"),
        "hgb_tuning_scores": res.get("hgb_tuning_scores"),
    }
with open(inner_tuning_summary_path, "w") as f:
    json.dump(serializable_tuning, f, indent=2)
print(f"Saved inner validation tuning summary to {inner_tuning_summary_path.name}")

# 5. Save Reconciliation Assertion Receipt JSON
reconciliation_receipt = {
    "task_id": "MP-FC-007",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "panel_v2_sha256": v2_sha256,
    "row_conservation_verified": f"{raw_total} = {acc_records} + {exact_dup} + {quar_conflict} + {quar_depth} + {quar_invalid}",
    "total_master_predictions": total_master_rows,
    "folds_evaluated": [f.fold_id for f in engine.folds],
    "models_evaluated": list(model_cols.keys()),
    "assertions_passed": [
        "panel_v2_sha256_match",
        "row_conservation_472385",
        "label_maturity_training_partitions",
        "outer_test_temporal_bounds",
        "tiny_smoke_test_serialization_np_allclose",
        "scorecard_row_count_reconciliation",
        "paired_naive_skill_ratio_of_sums",
    ],
    "status": "ALL_GATES_PASSED_BENCHMARK_SUCCESS",
}
reconciliation_receipt_path = reports_fc007_dir / "reconciliation_assertion_receipt.json"
with open(reconciliation_receipt_path, "w") as f:
    json.dump(reconciliation_receipt, f, indent=2)
print(f"Saved reconciliation receipt to {reconciliation_receipt_path.name}")

# 6. Save Output Manifest JSON
output_files = sorted(list(reports_fc007_dir.glob("*.csv")) + list(reports_fc007_dir.glob("*.json")))
manifest_records = []
for p in output_files:
    manifest_records.append({
        "filename": p.name,
        "path": str(p.relative_to(working_root)),
        "size_bytes": p.stat().st_size,
        "sha256": compute_sha256(p),
    })
# Include large artifacts stored under outputs
for p in [full_pred_parquet, full_pred_csv_gz]:
    if p.exists():
        manifest_records.append({
            "filename": p.name,
            "path": str(p.relative_to(working_root)),
            "size_bytes": p.stat().st_size,
            "sha256": compute_sha256(p),
        })
for p in models_dir.glob("*.joblib"):
    manifest_records.append({
        "filename": p.name,
        "path": str(p.relative_to(working_root)),
        "size_bytes": p.stat().st_size,
        "sha256": compute_sha256(p),
    })

manifest_path = reports_fc007_dir / "output_manifest.json"
with open(manifest_path, "w") as f:
    json.dump({"manifest": manifest_records}, f, indent=2)
print(f"Saved output manifest with {len(manifest_records)} artifacts to {manifest_path.name}")

print("\n=======================================================")
print("TASK MP-FC-007 BENCHMARK EXECUTION COMPLETED SUCCESSFULLY!")
print("=======================================================")
